In [35]:
import pandas as pd
import warnings
from dataclasses import replace

import numpy as np

from scripts.helpers import run_experiment
from scripts.propensity import get_propensity_scores
from variables.variables import *

df = pd.read_csv(DATAFRAME_PATH)
df

,Unnamed: 0.1,Unnamed: 0,index,RegistrationCode,PhysicalSleepTime,date,IntervalStart,PhysicalWakeTime,protein_g,fat_g,...,risks_taker_target_day,suffer_nerves_target_day,tense_or_highly_strung_target_day,tired_or_little_energy_fortnight_target_day,troubled_by_guilt_target_day,unenjoyment_uninterested_whole_week_target_day,work_satisfaction_target_day,worrier_target_day,worry_long_after_embarrassment_target_day,SleepTimeDifference_hours
0,0,6,6,10K_1007330152,2020-11-20 23:02:31,2020-11-20,2020-11-20 04:18:26,2020-11-21 04:18:26,89.919332,90.425066,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.0
1,1,9,9,10K_1008294272,2021-12-07 22:12:40,2021-12-07,2021-12-07 04:57:45,2021-12-08 04:57:45,96.700161,102.806969,...,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,23.0
2,2,24,24,10K_1015653163,2020-02-04 22:01:14,2020-02-04,2020-02-04 04:43:31,2020-02-05 04:43:31,51.040836,60.557672,...,0.0,0.0,1.0,1.0,1.0,1.0,4.0,1.0,0.0,24.0
3,3,25,25,10K_1015653163,2020-02-05 22:16:12,2020-02-05,2020-02-05 04:43:31,2020-02-06 04:27:00,93.889833,50.842479,...,0.0,0.0,1.0,1.0,1.0,1.0,4.0,1.0,0.0,24.0
4,4,30,30,10K_1018146705,2020-08-14 21:59:58,2020-08-14,2020-08-14 07:16:36,2020-08-15 07:16:36,51.092583,43.902700,...,-1.0,0.0,0.0,1.0,0.0,0.0,-1.0,0.0,1.0,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4788,4957,19683,19683,10K_9993849627,2022-05-14 21:07:03,2022-05-14,2022-05-14 03:13:27,2022-05-15 03:13:27,59.940134,131.524334,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0
4789,4958,19693,19693,10K_9995823183,2022-05-19 19:15:39,2022-05-19,2022-05-19 02:42:08,2022-05-20 02:42:08,137.959198,154.705081,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0
4790,4959,19696,19696,10K_9998420917,2023-07-29 20:04:02,2023-07-29,2023-07-29 04:03:22,2023-07-30 04:03:22,50.654954,46.736506,...,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,-1.0,23.0
4791,4960,19697,19697,10K_9998420917,2023-07-30 19:59:03,2023-07-30,2023-07-30 04:03:22,2023-07-31 03:07:03,58.417806,51.734639,...,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,-1.0,24.0


In [36]:
import numpy as np
import pandas as pd

def anonymize_preserve_distributions(df, seed=0):
    rng = np.random.default_rng(seed)
    fake = pd.DataFrame(index=df.index)

    for col in df.columns:
        s = df[col]

        # remember where NaNs were
        na_mask = s.isna()
        pool = s[~na_mask]

        if len(pool) == 0:
            fake[col] = s
            continue

        # sample from existing values
        sampled = rng.choice(pool.to_numpy(), size=len(s), replace=True)

        fake[col] = sampled
        fake.loc[na_mask, col] = np.nan

        # try to keep original dtype
        try:
            fake[col] = fake[col].astype(s.dtype)
        except Exception:
            pass

    return fake


df_fake = anonymize_preserve_distributions(df)

/var/folders/sy/fq837qjd5t3cjcxxpgyx52th0000gn/T/ipykernel_20013/4045497801.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  fake.loc[na_mask, col] = np.nan
/var/folders/sy/fq837qjd5t3cjcxxpgyx52th0000gn/T/ipykernel_20013/4045497801.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  fake.loc[na_mask, col] = np.nan
/var/folders/sy/fq837qjd5t3cjcxxpgyx52th0000gn/T/ipykernel_20013/4045497801.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  fake.loc[na_mask, col

In [37]:
df_fake

,Unnamed: 0.1,Unnamed: 0,index,RegistrationCode,PhysicalSleepTime,date,IntervalStart,PhysicalWakeTime,protein_g,fat_g,...,risks_taker_target_day,suffer_nerves_target_day,tense_or_highly_strung_target_day,tired_or_little_energy_fortnight_target_day,troubled_by_guilt_target_day,unenjoyment_uninterested_whole_week_target_day,work_satisfaction_target_day,worrier_target_day,worry_long_after_embarrassment_target_day,SleepTimeDifference_hours
0,4216,17715,7452,10K_9279377658,2021-08-09 21:03:17,2020-11-07,2023-12-02 05:31:26,2022-11-06 05:51:41,31.736843,97.408754,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0
1,3152,14454,4701,10K_6405759997,2021-02-17 22:43:46,2022-09-29,2023-02-15 04:31:13,2021-04-09 03:58:53,47.087359,128.240940,...,0.0,0.0,0.0,0.0,1.0,0.0,5.0,1.0,0.0,24.0
2,2526,3057,6451,10K_7584685318,2021-09-19 21:02:11,2023-08-02,2021-12-12 04:04:08,2020-07-31 02:37:06,105.815858,86.674691,...,1.0,0.0,0.0,2.0,1.0,0.0,5.0,1.0,1.0,23.0
3,1333,2287,2456,10K_7232691006,2021-09-03 23:19:05,2021-02-18,2021-10-18 03:47:00,2020-11-29 05:32:04,24.563359,56.432437,...,1.0,0.0,0.0,2.0,1.0,1.0,5.0,0.0,0.0,23.0
4,1520,11981,10640,10K_2235515519,2022-03-28 19:32:44,2023-06-27,2021-08-09 03:44:41,2021-02-21 03:37:55,157.895712,124.163906,...,0.0,0.0,1.0,1.0,0.0,0.0,5.0,1.0,0.0,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4788,1365,1212,7262,10K_8914217155,2020-06-08 19:36:38,2022-07-09,2022-12-27 04:13:55,2020-02-08 10:36:25,50.775900,129.250140,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0
4789,2083,7636,5310,10K_9906173268,2023-03-25 22:03:32,2022-07-15,2022-10-01 04:46:05,2023-03-01 04:02:57,96.441347,47.952264,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.0
4790,4535,15717,18317,10K_3428687646,2023-03-19 19:26:00,2023-05-24,2020-06-23 04:35:00,2021-07-10 03:04:15,43.242328,79.660502,...,-1.0,0.0,0.0,1.0,0.0,0.0,-1.0,0.0,0.0,24.0
4791,811,3769,10063,10K_6699134764,2020-03-21 21:36:47,2022-04-24,2022-07-06 03:27:31,2020-09-15 02:23:13,54.088557,85.620783,...,0.0,0.0,1.0,1.0,0.0,0.0,2.0,0.0,0.0,24.0


In [38]:
df_fake.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'index', 'RegistrationCode',
       'PhysicalSleepTime', 'date', 'IntervalStart', 'PhysicalWakeTime',
       'protein_g', 'fat_g',
       ...
       'risks_taker_target_day', 'suffer_nerves_target_day',
       'tense_or_highly_strung_target_day',
       'tired_or_little_energy_fortnight_target_day',
       'troubled_by_guilt_target_day',
       'unenjoyment_uninterested_whole_week_target_day',
       'work_satisfaction_target_day', 'worrier_target_day',
       'worry_long_after_embarrassment_target_day',
       'SleepTimeDifference_hours'],
      dtype='object', length=1420)

In [39]:
with open("columns.txt", "w") as f:
    for col in df.columns:
        f.write(f"{col}\n")

In [40]:
df_fake.to_csv('mock_data/df.csv')

In [41]:
import pandas as pd 

df_fake = pd.read_csv('mock_data/df.csv')

In [42]:
features = [
        "RegistrationCode", #"id",
        "protein_g",
        "fat_g",
        "carbs_g",
        "sugars_g",
        "fiber_g",
        "sat_fat_g",
        "poly_fat_g",
        "mono_fat_g",
        "omega3_total_g",
        "omega3_to_6",
        "magnesium_mg",
        "potassium_mg",
        "calcium_mg",
        "vitamin_d_ug",
        "vitamin_b6_mg",
        "vitamin_b12_ug",
        "zinc_mg",
        "sodium_mg",
        "tryptophan",
        "folate",
        "choline",
        "vitamin_c",
        "vitamin_e",
        "hour_first_meal",
        "hour_last_meal",
        "hours_to_sleep",
        "eating_window_h",
        "night_calories",
        "night_calories_pct",
        "night_carbs_g",
        "night_protein_g",
        "night_fat_g",
        "night_protein_pct_energy",
        "night_carbs_pct_energy",
        "night_fat_pct_energy",
        "last_meal_calories",
        "last_meal_calories_pct",
        "unique_foods_count",
        "unique_plant_based_foods_count",
        "caffeine_late_mg",
        "hours_since_last_caffeine",
        "whole_foods_categories_energy",
        "plant_based_whole_foods_categories_energy",
        "whole_dairy_categories_energy",
        "processed_categories_energy",
        "total_energy_kcal",
        "whole_food_categories_ratio",
        "plant_based_whole_foods_ratio",
        "whole_dairy_categories_ratio",
        "processed_categories_ratio",
        "prot_pct",
        "fat_pct",
        "carb_pct",
        "sleep_hour",
        "percent_of_deep_sleep_time",
        "percent_of_rem_sleep_time",
        "percent_of_light_sleep_time",
        "number_of_wakes",
        "heart_rate_mean_during_sleep",
        "neurokit_hrv_time_rmssd_during_nrem",
        "sleep_efficiency",
        "total_sleep_time_minutes",
        "sleep_latency_minutes",
        "total_wake_time_after_sleep_onset_minutes",
        "age",
        "bmi",
        "hand_grip_left",
        "hand_grip_right",
        "height",
        "hips",
        "sitting_blood_pressure_diastolic",
        "sitting_blood_pressure_pulse_rate",
        "sitting_blood_pressure_systolic",
        "waist",
        "weight",
        "whr",
        "body_comp_android_tissue_percent_fat",
        "bt__hba1c",
        "bt__glucose",
        "bt__insulin",
        "bt__total_cholesterol",
        "bt__ldl_cholesterol",
        "bt__hdl_cholesterol",
        "bt__triglycerides",
        "bt__cholesterol_hdl_ratio",
        "bt__egfr",
        "bt__creatinine",
        "bt__crp_hs",
        "bt__vitamin_d",
        "bt__albumin",
        "bt__ferritin",
        "bt__uric_acid",

        "weekday",
        "month",
        "gender",

        "active_lifestyle",
        "walking_for_pleasure",
        "other_physical_activity",
        "exercise",
        "walk",
        "no_activity_daily",
        "smoking_less_than_one_per_day",
        "smoking_more_than_one_per_day",
        "smoking_unknown",
        "never_eat_dairy",
        "never_eat_eggs",
        "no_dietary_restrictions",
        "never_eat_sugar",
        "never_eat_wheat",
        "living_with_spouse",
        "living_with_partners",
        "living_with_chilren",
        "has_cat",
        "has_dog",
        "smoking_cigarettes",
        "uses_bike",
        "uses_car",
        "uses_public_transportation",
        "walks_to_work",
        "depressed_hopeless_fortnight",
        "depressed_whole_week",
        "family_relationship_satisfaction",
        "fed-up_often",
        "feeling_lonley_often",
        "fidgety_or_restless_fortnight",
        "financial_situation_satisfaction",
        "friendships_satisfaction",
        "happiness_level",
        "health_satisfaction",
        "hyper_least_two_days",
        "irritable_person",
        "irritablilty_leading_to_argumens",
        "little_interest_or_pleasure_fortnight",
        "miserable_no_reason",
        "mood_swings",
        "nerves_anxiety_tension_depression_doctor",
        "nerves_anxiety_tension_depression_psychiatrist",
        "nervous_person",
        "rds_score",
        "risks_taker",
        "suffer_nerves",
        "tense_or_highly_strung",
        "tired_or_little_energy_fortnight",
        "troubled_by_guilt",
        "unenjoyment_uninterested_whole_week",
        "work_satisfaction",
        "worrier",
        "worry_long_after_embarrassment"
    ]

In [43]:
sleep_and_food = [
        "protein_g",
        "fat_g",
        "carbs_g",
        "sugars_g",
        "fiber_g",
        "sat_fat_g",
        "poly_fat_g",
        "mono_fat_g",
        "omega3_total_g",
        "omega3_to_6",
        "magnesium_mg",
        "potassium_mg",
        "calcium_mg",
        "vitamin_d_ug",
        "vitamin_b6_mg",
        "vitamin_b12_ug",
        "zinc_mg",
        "sodium_mg",
        "tryptophan",
        "folate",
        "choline",
        "vitamin_c",
        "vitamin_e",
        "hour_first_meal",
        "hour_last_meal",
        "hours_to_sleep",
        "eating_window_h",
        "night_calories",
        "night_calories_pct",
        "night_carbs_g",
        "night_protein_g",
        "night_fat_g",
        "night_protein_pct_energy",
        "night_carbs_pct_energy",
        "night_fat_pct_energy",
        "last_meal_calories",
        "last_meal_calories_pct",
        "unique_foods_count",
        "unique_plant_based_foods_count",
        "caffeine_late_mg",
        "hours_since_last_caffeine",
        "whole_foods_categories_energy",
        "plant_based_whole_foods_categories_energy",
        "whole_dairy_categories_energy",
        "processed_categories_energy",
        "total_energy_kcal",
        "whole_food_categories_ratio",
        "plant_based_whole_foods_ratio",
        "whole_dairy_categories_ratio",
        "processed_categories_ratio",
        "prot_pct",
        "fat_pct",
        "carb_pct",
        "sleep_hour",
        "percent_of_deep_sleep_time",
        "percent_of_rem_sleep_time",
        "percent_of_light_sleep_time",
        "number_of_wakes",
        "heart_rate_mean_during_sleep",
        "neurokit_hrv_time_rmssd_during_nrem",
        "sleep_efficiency",
        "total_sleep_time_minutes",
        "sleep_latency_minutes",
        "total_wake_time_after_sleep_onset_minutes",
        "quality_score_heart_rate",
        "percent_of_supine_sleep",
        "total_left_sleep_time_minutes",
]

In [44]:
features_final = features + [s + '_target_day' for s in sleep_and_food]

In [45]:
df_fake[features_final].sample(2500).to_csv('mock_data/df.csv')